In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_SDY2867.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,SDY2867.SUB389725,0,ENSG00000000003,2.0,0.038336,Whole blood
1,SDY2867.SUB389727,-14,ENSG00000000003,10.0,0.200864,Whole blood
2,SDY2867.SUB389692,1,ENSG00000000003,16.0,0.964728,Whole blood
3,SDY2867.SUB389670,28,ENSG00000000003,17.0,0.670581,Whole blood
4,SDY2867.SUB389681,0,ENSG00000000003,2.0,0.094223,Whole blood


In [3]:
# raw_count has actual values (no nulls); material is uniformly 'Whole blood'
# SDY2867 has 5 timepoints: [-14, 0, 1, 7, 28] vs 2 for 2020_UGA
# Drop material before pivoting; drop raw_count to keep tpm_count only, consistent with 2019/2020 pipeline
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: <StringArray>
['Whole blood']
Length: 1, dtype: str
timepoints: [np.int64(-14), np.int64(0), np.int64(1), np.int64(7), np.int64(28)]
participants: 73


In [4]:
# Drop material (entirely 'Whole blood'); drop raw_count to match pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,SDY2867.SUB389725,0,ENSG00000000003,0.038336
1,SDY2867.SUB389727,-14,ENSG00000000003,0.200864
2,SDY2867.SUB389692,1,ENSG00000000003,0.964728
3,SDY2867.SUB389670,28,ENSG00000000003,0.670581
4,SDY2867.SUB389681,0,ENSG00000000003,0.094223


In [5]:
# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='ensembl_gene_id',
    values='tpm_count'
)
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()
print(df_pivot.shape)
df_pivot.head()

(361, 54904)


,participant_id,timepoint,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,...,ENSG00000310527,ENSG00000310529,ENSG00000310530,ENSG00000310532,ENSG00000310533,ENSG00000310534,ENSG00000310535,ENSG00000310536,ENSG00000310537,ENSG00000310539
0,SDY2867.SUB389659,-14,0.236575,0.0,30.356589,8.274846,1.371220,403.127865,0.931135,15.219241,...,51.852476,0.0,0.0,0.0,7.464742,1.516073,6.996225,1.199348,0.108096,8.322609
1,SDY2867.SUB389659,0,0.375210,0.0,37.867705,8.410112,1.897157,452.649272,1.293022,12.468978,...,54.175804,0.0,0.0,0.0,6.709422,1.745047,6.494206,1.211741,1.255793,8.559439
2,SDY2867.SUB389659,1,0.441595,0.0,36.836252,7.451985,1.395465,474.281275,0.964826,14.716382,...,40.224780,0.0,0.0,0.0,5.946249,3.568315,5.105175,1.650630,1.119001,4.793304
3,SDY2867.SUB389659,7,0.357179,0.0,35.201094,8.384532,1.997252,414.366967,1.137518,14.075322,...,55.040795,0.0,0.0,0.0,6.686735,0.993302,6.994508,0.836473,0.940114,6.580190
4,SDY2867.SUB389659,28,0.171582,0.0,29.141576,7.167474,1.467131,519.091736,0.760005,11.430638,...,40.093229,0.0,0.0,0.0,4.035190,0.857613,7.787015,0.277979,0.379442,3.724106


In [7]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/transcriptomics_SDY2867_cleaned.csv', index=False)